# 🎯 RS-LiDAR & LiDAR: Chấm Lại Điểm GenEval (Mask2Former Swin-S)
### Bảng Tổng Hợp 6 Dòng Chuẩn Khoa Học Cho Cả 3 Settings:
1. **SD v1.5 (DDIM-50, $\eta=0.0$)**: Vanilla LiDAR vs RS-LiDAR ($\sigma=1.0, M=4$)
2. **SD v1.5 (DDPM-100, $\eta=1.0$)**: Vanilla LiDAR vs RS-LiDAR ($\sigma=1.0, M=4$)
3. **SDXL 2.6B (DDPM-100, $\eta=1.0$)**: Vanilla LiDAR vs RS-LiDAR ($\sigma=1.0, M=4$)

---
### 📌 Cơ Chế Hoạt Động (Chỉ Cần Bấm Run All):
- **Gắn kết Google Drive cá nhân** (`/content/drive/MyDrive/RS-LiDAR`).
- **Tự động tìm đúng 6 thư mục thực nghiệm** tương ứng với 6 dòng kết quả.
- **Đọc trực tiếp `final_metrics.json`** trong từng thư mục để lấy: **ImageReward, CLIP-Score, HPS v2.1**.
- **Chấm lại GenEval** bằng mô hình chuẩn bài báo gốc **Mask2Former Swin-S COCO** (`facebook/mask2former-swin-small-coco-instance`, mAP ~52%).
- **Đầu ra duy nhất**: Một bảng tổng kết khoa học gồm **đúng 6 dòng** so sánh trực diện giữa Vanilla LiDAR và RS-LiDAR trên cả 3 settings!


## 1. Cài Đặt Môi Trường & Kiểm Tra GPU

In [ ]:
# @title 📁 Gắn kết Google Drive cá nhân & Khởi tạo đường dẫn cứng
from google.colab import drive
drive.mount('/content/drive')

base_drive = '/content/drive/MyDrive' if os.path.exists('/content/drive/MyDrive') else '/content/drive/My Drive'
DRIVE_DIR = f"{base_drive}/RS-LiDAR"
TARGET_BASE = f"{DRIVE_DIR}/Target_samples"

assert os.path.exists(DRIVE_DIR), f"❌ Không tìm thấy thư mục RS-LiDAR tại: {DRIVE_DIR}!"
assert os.path.exists(TARGET_BASE), f"❌ Không tìm thấy thư mục Target_samples tại: {TARGET_BASE}!"

print(f"✅ Đã kết nối thành công với Google Drive cá nhân!")
print(f"📁 Thư mục gốc: {DRIVE_DIR}")
print(f"🎯 Thư mục Target_samples: {TARGET_BASE}")


## 2. Gắn Kết Google Drive Cá Nhân & Xác Định Thư Mục Dữ Liệu

In [ ]:
# @title 🔍 Cấu hình đường dẫn cứng cho đúng 6 thư mục & Kiểm tra tồn tại

# 6 thư mục cố định chuẩn xác cần chấm điểm:
TARGET_EXPERIMENTS = [
    # --- SETTING 1: SD v1.5 DDIM-50 (eta=0.0) ---
    {
        "setting": "SD v1.5 DDIM-50",
        "method": "Vanilla LiDAR",
        "folder": "LiDAR_SD15_DPM5_n50_DDIM50_s12.5_lmbda5000_seed100_A100"
    },
    {
        "setting": "SD v1.5 DDIM-50",
        "method": "RS-LiDAR (σ=1.0)",
        "folder": "RSLiDAR_SD15_DPM5_n50_sig1.0_M4_DDIM50_s12.5_lmbda5000_seed100_A100"
    },

    # --- SETTING 2: SD v1.5 DDPM-100 (eta=1.0) ---
    {
        "setting": "SD v1.5 DDPM-100",
        "method": "Vanilla LiDAR",
        "folder": "LiDAR_SD15_DPM5_n50_DDPM100_s12.5_lmbda5000_seed100_A100"
    },
    {
        "setting": "SD v1.5 DDPM-100",
        "method": "RS-LiDAR (σ=1.0)",
        "folder": "RSLiDAR_SD15_DPM5_n50_sig1.0_M4_DDPM100_s12.5_lmbda5000_seed100_A100"
    },

    # --- SETTING 3: SDXL DDPM-100 (eta=1.0) ---
    {
        "setting": "SDXL DDPM-100",
        "method": "Vanilla LiDAR",
        "folder": "LiDAR_SDXL_DMD1_Step1_n100_DDPM100_s8.0_lmbda5000_seed100_A100"
    },
    {
        "setting": "SDXL DDPM-100",
        "method": "RS-LiDAR (σ=1.0)",
        "folder": "RSLiDAR_SDXL_DMD1_Step1_n100_sig1.0_M4_DDPM100_s8.0_lmbda5000_seed100_A100"
    }
]

print("=" * 95)
print("🔍 KIỂM TRA SỰ TỒN TẠI CỦA ĐÚNG 6 THƯ MỤC TRƯỚC KHI CHẤM:")
print("=" * 95)

for i, exp in enumerate(TARGET_EXPERIMENTS, 1):
    exp["path"] = os.path.join(TARGET_BASE, exp["folder"])
    exists = os.path.exists(exp["path"]) and os.path.isdir(exp["path"])
    prompt_count = len(glob.glob(f"{exp['path']}/[0-9]*")) if exists else 0
    exp["prompt_count"] = prompt_count
    
    status_text = "✅ TỒN TẠI" if exists else "❌ KHÔNG TÌM THẤY"
    print(f"{i}. {status_text} | [{exp['setting']}] {exp['method']}:")
    print(f"   📁 {exp['path']} ({prompt_count}/553 prompts)")
    
    # Dừng lại ngay nếu có thư mục nào không tồn tại để người dùng biết chính xác
    assert exists, f"❌ Lỗi: Không tìm thấy thư mục: {exp['path']} trên Drive!"

print("=" * 95)
print("🎉 XÁC NHẬN 100%: CẢ 6 THƯ MỤC ĐỀU TỒN TẠI ĐẦY ĐỦ TRÊN DRIVE! SẴN SÀNG CHẤM ĐIỂM.")
print("=" * 95)


# @title 🚀 BẮT ĐẦU CHẤM ĐIỂM GENEVAL CHO ĐÚNG 6 THỰC NGHIỆM

FORCE_RESCORE = True  # True: Chấm lại toàn bộ đè lên file cũ | False: Tận dụng file geneval_summary.csv nếu đã có

def evaluate_geneval_for_folder(target_dir, exp_name):
    """Đánh giá toàn bộ ảnh trong target_dir bằng Mask2Former Swin-S và lưu geneval_summary.csv"""
    assert os.path.exists(target_dir), f"❌ Thư mục không tồn tại: {target_dir}"

    geneval_csv_path = f"{target_dir}/geneval_summary.csv"
    if not FORCE_RESCORE and os.path.exists(geneval_csv_path):
        try:
            df_exist = pd.read_csv(geneval_csv_path)
            for _, r in df_exist.iterrows():
                if 'OVERALL' in str(r.iloc[0]).upper():
                    print(f"⚡ Tận dụng điểm GenEval đã có sẵn cho {exp_name}: {float(r.iloc[2]):.4f}")
                    return float(r.iloc[2])
        except Exception:
            pass

    # 1. Quét ảnh trong các thư mục prompt
    prompt_dir_map = {}
    for p_dir in glob.glob(f"{target_dir}/[0-9]*"):
        try:
            p_idx = int(os.path.basename(p_dir))
        except ValueError:
            continue
        imgs = sorted(glob.glob(f"{p_dir}/samples/*.png")) or [f for f in glob.glob(f"{p_dir}/*.png") if not f.endswith("grid.png")]
        if imgs:
            prompt_dir_map[p_idx] = imgs

    if not prompt_dir_map:
        print(f"⚠️ Không tìm thấy ảnh hợp lệ trong: {exp_name}")
        return None

    print(f"\n" + "="*75)
    print(f"🎯 Đang chấm GenEval (Mask2Former Swin-S): {exp_name}")
    print(f"   Số prompt có ảnh: {len(prompt_dir_map)}/553")
    print("="*75)

    task_results = {'single_object': [], 'two_object': [], 'counting': [], 'colors': [], 'position': [], 'color_attr': []}
    
    for p_idx in tqdm(sorted(prompt_dir_map.keys()), desc=f"Chấm điểm: {exp_name[:30]}..."):
        if p_idx >= len(prompts_meta):
            continue
        meta = prompts_meta[p_idx]
        tag = meta.get('tag', 'single_object')
        if tag not in task_results:
            continue

        imgs = prompt_dir_map[p_idx]
        prompt_scores = []
        for img_path in imgs:
            try:
                img = Image.open(img_path).convert('RGB')
                inputs = image_processor(images=img, return_tensors="pt").to(device)
                with torch.inference_mode():
                    outputs = detector(**inputs)

                # Instance Segmentation threshold = 0.5 chuẩn Mask2Former
                results = image_processor.post_process_instance_segmentation(
                    outputs, target_sizes=[img.size[::-1]], threshold=0.5
                )[0]
                segmentation = results["segmentation"].detach().cpu().numpy()
                segments_info = results["segments_info"]

                detected_objects = []
                for seg in segments_info:
                    c_name = id2label[seg["label_id"]].lower()
                    mask = (segmentation == seg["id"])
                    y_indices, x_indices = np.where(mask)
                    if len(x_indices) > 0 and len(y_indices) > 0:
                        x1, x2 = float(np.min(x_indices)), float(np.max(x_indices))
                        y1, y2 = float(np.min(y_indices)), float(np.max(y_indices))
                        box = [x1, y1, x2, y2]
                        center_x = (x1 + x2) / 2.0
                        center_y = (y1 + y2) / 2.0

                        if (x2 - x1 > 12 and y2 - y1 > 12):
                            crop = img.crop((max(0, x1), max(0, y1), min(img.width, x2), min(img.height, y2)))
                            pred_color = classify_crop_color(crop)
                        else:
                            pred_color = 'unknown'

                        detected_objects.append({
                            'class': c_name,
                            'box': box,
                            'center_x': center_x,
                            'center_y': center_y,
                            'color': pred_color
                        })

                # Đánh giá theo rule bài báo GenEval
                success = False
                includes = meta.get('include', [])
                if tag == 'single_object':
                    req_cls = includes[0]['class'].lower()
                    success = any(req_cls in obj['class'] or obj['class'] in req_cls for obj in detected_objects)
                elif tag == 'two_object':
                    req1, req2 = includes[0]['class'].lower(), includes[1]['class'].lower()
                    success = any(req1 in obj['class'] or obj['class'] in req1 for obj in detected_objects) and \
                               any(req2 in obj['class'] or obj['class'] in req2 for obj in detected_objects)
                elif tag == 'counting':
                    req_cls, target_count = includes[0]['class'].lower(), includes[0]['count']
                    found_count = sum(1 for obj in detected_objects if req_cls in obj['class'] or obj['class'] in req_cls)
                    success = (found_count == target_count)
                elif tag == 'colors':
                    req_cls, req_color = includes[0]['class'].lower(), includes[0]['color'].lower()
                    success = any((req_cls in obj['class'] or obj['class'] in req_cls) and (obj['color'] == req_color) for obj in detected_objects)
                elif tag == 'position':
                    req1, req2 = includes[0]['class'].lower(), includes[1]['class'].lower()
                    pos_type = includes[1].get('position', ['right of', 0])[0]
                    o1_list = [o for o in detected_objects if req1 in o['class'] or o['class'] in req1]
                    o2_list = [o for o in detected_objects if req2 in o['class'] or o['class'] in req2]
                    if o1_list and o2_list:
                        o1, o2 = o1_list[0], o2_list[0]
                        if 'right' in pos_type: success = (o2['center_x'] > o1['center_x'])
                        elif 'left' in pos_type: success = (o2['center_x'] < o1['center_x'])
                        elif 'above' in pos_type or 'top' in pos_type: success = (o2['center_y'] < o1['center_y'])
                        elif 'below' in pos_type or 'bottom' in pos_type: success = (o2['center_y'] > o1['center_y'])
                        else: success = True
                elif tag == 'color_attr':
                    success = all(any((inc['class'].lower() in obj['class'] or obj['class'] in inc['class'].lower()) and (obj['color'] == inc['color'].lower()) for obj in detected_objects) for inc in includes)

                prompt_scores.append(1.0 if success else 0.0)
            except Exception as e:
                pass

        if prompt_scores:
            task_results[tag].append(np.mean(prompt_scores))

    # Tổng hợp kết quả từng task
    summary_rows = []
    all_means = []
    for t_name, scores in task_results.items():
        mean_val = np.mean(scores) if scores else 0.0
        if scores: all_means.append(mean_val)
        summary_rows.append({'Nhiệm Vụ (Task)': t_name, 'Số Lượng Prompt': len(scores), 'Độ Chính Xác (Accuracy ↑)': f"{mean_val:.4f}"})

    overall_geneval = float(np.mean(all_means)) if all_means else 0.0
    summary_rows.append({'Nhiệm Vụ (Task)': '🔥 OVERALL GENEVAL BENCHMARK', 'Số Lượng Prompt': sum(len(s) for s in task_results.values()), 'Độ Chính Xác (Accuracy ↑)': f"{overall_geneval:.4f}"})
    
    df_geneval = pd.DataFrame(summary_rows)
    df_geneval.to_csv(geneval_csv_path, index=False)
    print(f"💾 Đã lưu bảng điểm GenEval chuẩn Mask2Former tại: {geneval_csv_path}")
    print(f"⭐ ĐIỂM OVERALL GENEVAL: {overall_geneval:.4f}")
    return overall_geneval

# Thực thi chấm điểm tuần tự cho đúng 6 thực nghiệm
for exp in TARGET_EXPERIMENTS:
    exp["geneval_score"] = evaluate_geneval_for_folder(exp["path"], exp["folder"])

print("\n✅ ĐÃ HOÀN TẤT CHẤM ĐIỂM GENEVAL CHO TOÀN BỘ 6 THỰC NGHIỆM!")


In [ ]:
# @title 📊 XUẤT BẢNG TỔNG HỢP KHOA HỌC CHUẨN ĐÚNG 6 DÒNG

def extract_metrics(folder_path):
    """Đọc các chỉ số ImageReward, CLIP, HPS v2.1 trực tiếp từ final_metrics.json"""
    if not folder_path or not os.path.exists(folder_path):
        return {"ir": "N/A", "clip": "N/A", "hps": "N/A", "count": 0}

    # 1. Đọc trực tiếp từ final_metrics.json (< 0.01s)
    fm_json = f"{folder_path}/final_metrics.json"
    if os.path.exists(fm_json):
        try:
            with open(fm_json, 'r') as f:
                d = json.load(f)
                ir_val = d.get('ImageReward', d.get('ir'))
                clip_val = d.get('CLIP', d.get('clip'))
                hps_val = d.get('HPS', d.get('hps'))
                count = d.get('total_prompts', len(glob.glob(f"{folder_path}/[0-9]*")))
                return {
                    "ir": f"{float(ir_val):.4f}" if ir_val is not None else "N/A",
                    "clip": f"{float(clip_val):.4f}" if clip_val is not None else "N/A",
                    "hps": f"{float(hps_val):.4f}" if hps_val is not None else "N/A",
                    "count": count
                }
        except Exception:
            pass

    # 2. Fallback quét results.json nếu final_metrics.json chưa được sinh
    r_files = glob.glob(f"{folder_path}/[0-9]*/results.json")
    if r_files:
        irs, clips, hpss = [], [], []
        for rf in r_files:
            try:
                with open(rf, 'r') as f:
                    d = json.load(f)
                if 'image_reward' in d: irs.append(d['image_reward'])
                if 'clip_score' in d: clips.append(d['clip_score'])
                if 'hps_score' in d: hpss.append(d['hps_score'])
            except Exception:
                pass
        return {
            "ir": f"{np.mean(irs):.4f}" if irs else "N/A",
            "clip": f"{np.mean(clips):.4f}" if clips else "N/A",
            "hps": f"{np.mean(hpss):.4f}" if hpss else "N/A",
            "count": len(r_files)
        }

    return {"ir": "N/A", "clip": "N/A", "hps": "N/A", "count": 0}

# Xây dựng đúng 6 dòng kết quả chuẩn
table_rows = []

for exp in TARGET_EXPERIMENTS:
    folder_path = exp["path"]
    metrics = extract_metrics(folder_path)
    
    # Lấy điểm GenEval chuẩn vừa chấm
    ge_val = exp.get("geneval_score")
    if ge_val is None and folder_path:
        ge_csv = f"{folder_path}/geneval_summary.csv"
        if os.path.exists(ge_csv):
            try:
                df_ge = pd.read_csv(ge_csv)
                for _, r in df_ge.iterrows():
                    if 'OVERALL' in str(r.iloc[0]).upper():
                        ge_val = float(r.iloc[2])
                        break
            except Exception:
                pass
                
    ge_str = f"{ge_val:.4f}" if ge_val is not None else "N/A"
    method_name = f"🔥 {exp['method']}" if "RS-LiDAR" in exp["method"] else exp["method"]
    
    table_rows.append({
        "Setting (Cấu Hình)": exp["setting"],
        "Phương Pháp": method_name,
        "ImageReward ↑": metrics["ir"],
        "CLIP-Score ↑": metrics["clip"],
        "HPS v2.1 ↑": metrics["hps"],
        "GenEval ↑": ge_str,
        "Số Prompts": f"{metrics['count']}/553",
        "Thư Mục Dữ Liệu": exp["folder"]
    })

final_6rows_df = pd.DataFrame(table_rows)

print("\n" + "="*110)
print("📊 BẢNG KẾT QUẢ KHOA HỌC CHUẨN 6 DÒNG (3 SETTINGS x 2 METHODS)")
print("   Mô hình đánh giá GenEval: Mask2Former Swin-S COCO (Threshold 0.5) + CLIP ViT-B/32")
print("="*110)
display(final_6rows_df)

# Tính toán mức tăng trưởng Delta (Δ) của RS-LiDAR so với Vanilla LiDAR trên từng Setting
print("\n" + "="*110)
print("📈 PHÂN TÍCH MỨC TĂNG TRƯỞNG (Δ GAIN) CỦA RS-LiDAR SO VỚI VANILLA LIDAR:")
print("="*110)

for s_idx in [0, 2, 4]:
    row_lidar = table_rows[s_idx]
    row_rslidar = table_rows[s_idx + 1]
    setting_name = row_lidar["Setting (Cấu Hình)"]
    
    try:
        delta_ir = float(row_rslidar["ImageReward ↑"]) - float(row_lidar["ImageReward ↑"])
        pct_ir = (delta_ir / abs(float(row_lidar["ImageReward ↑"]))) * 100
        delta_clip = float(row_rslidar["CLIP-Score ↑"]) - float(row_lidar["CLIP-Score ↑"])
        delta_hps = float(row_rslidar["HPS v2.1 ↑"]) - float(row_lidar["HPS v2.1 ↑"])
        delta_ge = float(row_rslidar["GenEval ↑"]) - float(row_lidar["GenEval ↑"])
        
        print(f"🔹 [{setting_name}]:")
        print(f"   • ImageReward: {delta_ir:+.4f} ({pct_ir:+.2f}%)")
        print(f"   • GenEval:     {delta_ge:+.4f}")
        print(f"   • CLIP-Score:  {delta_clip:+.4f}")
        print(f"   • HPS v2.1:    {delta_hps:+.4f}")
    except Exception as e:
        print(f"🔹 [{setting_name}]: Chưa đủ dữ liệu số để tính delta ({e})")
    print("-" * 70)

# Lưu ra Google Drive
master_csv_path = f"{TARGET_BASE}/table2_master_6rows_summary.csv"
final_6rows_df.to_csv(master_csv_path, index=False)
print(f"\n💾 Đã lưu bảng tổng hợp 6 dòng ra Google Drive tại:\n   {master_csv_path}")
print("="*110)


## 4. Tự Động Định Vị Chính Xác 6 Thư Mục Thực Nghiệm (3 Settings x 2 Methods)

In [ ]:
# @title 🔍 Tự động tìm đúng 6 thư mục tương ứng với 3 settings chuẩn Bảng 2

# Định nghĩa cấu hình mục tiêu cho đúng 6 dòng
TARGET_CONFIGS = [
    # Setting 1: SD 1.5 DDIM-50 (eta=0.0)
    {
        "setting": "SD v1.5 DDIM-50",
        "method": "Vanilla LiDAR",
        "patterns": [
            "LiDAR_SD15_*DDIM*",
            "LiDAR_SD15_*Step50*eta0*",
            "LiDAR_SD15_*Step5_n50_DDIM50*",
            "*SD15*DDIM*LiDAR*",
            "*SD15*DDIM*"
        ],
        "exclude": ["RSLiDAR", "sig", "smoothing"]
    },
    {
        "setting": "SD v1.5 DDIM-50",
        "method": "RS-LiDAR (σ=1.0)",
        "patterns": [
            "RSLiDAR_SD15_*sig1*DDIM*",
            "RSLiDAR_SD15_*sig1.0*DDIM*",
            "RSLiDAR_SD15_*DDIM*",
            "*RSLiDAR*SD15*DDIM*"
        ],
        "exclude": []
    },

    # Setting 2: SD 1.5 DDPM-100 (eta=1.0)
    {
        "setting": "SD v1.5 DDPM-100",
        "method": "Vanilla LiDAR",
        "patterns": [
            "LiDAR_SD15_*DDPM*",
            "LiDAR_SD15_*Step100*",
            "LiDAR_SD15_*Step5_n50_DDPM100*",
            "*SD15*DDPM*LiDAR*",
            "*SD15*DDPM*"
        ],
        "exclude": ["RSLiDAR", "sig", "smoothing"]
    },
    {
        "setting": "SD v1.5 DDPM-100",
        "method": "RS-LiDAR (σ=1.0)",
        "patterns": [
            "RSLiDAR_SD15_*sig1*DDPM*",
            "RSLiDAR_SD15_*sig1.0*DDPM*",
            "RSLiDAR_SD15_*DDPM*",
            "*RSLiDAR*SD15*DDPM*"
        ],
        "exclude": []
    },

    # Setting 3: SDXL DDPM-100 (eta=1.0)
    {
        "setting": "SDXL DDPM-100",
        "method": "Vanilla LiDAR",
        "patterns": [
            "LiDAR_SDXL_*DDPM*",
            "LiDAR_SDXL_DMD1_*",
            "LiDAR_SDXL*",
            "*SDXL*LiDAR*"
        ],
        "exclude": ["RSLiDAR", "sig", "smoothing"]
    },
    {
        "setting": "SDXL DDPM-100",
        "method": "RS-LiDAR (σ=1.0)",
        "patterns": [
            "RSLiDAR_SDXL_*sig1*DDPM*",
            "RSLiDAR_SDXL_*sig1.0*DDPM*",
            "RSLiDAR_SDXL_*DDPM*",
            "RSLiDAR_SDXL_DMD1_*",
            "RSLiDAR_SDXL*"
        ],
        "exclude": []
    }
]

matched_experiments = []

for cfg in TARGET_CONFIGS:
    found_dir = None
    for pat in cfg["patterns"]:
        candidates = sorted(glob.glob(f"{TARGET_BASE}/{pat}"))
        for cand in candidates:
            cand_name = os.path.basename(cand)
            # Kiểm tra exclude
            if any(ex.lower() in cand_name.lower() for ex in cfg["exclude"]):
                continue
            # Kiểm tra có ảnh
            p_dirs = glob.glob(f"{cand}/[0-9]*")
            if len(p_dirs) >= 10:
                found_dir = cand
                break
        if found_dir:
            break
            
    matched_experiments.append({
        "setting": cfg["setting"],
        "method": cfg["method"],
        "path": found_dir,
        "name": os.path.basename(found_dir) if found_dir else "KHÔNG TÌM THẤY",
        "prompt_count": len(glob.glob(f"{found_dir}/[0-9]*")) if found_dir else 0
    })

print("=" * 95)
print("📋 DANH SÁCH 6 THỰC NGHIỆM ĐƯỢC TỰ ĐỘNG KHỚP TRÊN DRIVE CÁ NHÂN:")
print("=" * 95)
for i, exp in enumerate(matched_experiments, 1):
    status_icon = "✅" if exp["path"] else "❌"
    print(f"{i:1d}. {status_icon} [{exp['setting']}] {exp['method']}:")
    print(f"      📁 Thư mục: {exp['name']} ({exp['prompt_count']}/553 prompts)")
print("=" * 95)


## 5. Chấm Điểm GenEval Bằng Mask2Former Chuẩn Bài Báo Cho 6 Thực Nghiệm

In [ ]:
# @title 🚀 BẮT ĐẦU CHẤM ĐIỂM GENEVAL CHO 6 THỰC NGHIỆM

# Cho phép chọn: True = Chấm lại toàn bộ đè lên file cũ | False = Tận dụng file geneval_summary.csv nếu đã có
FORCE_RESCORE = True

def evaluate_geneval_for_folder(target_dir, exp_name):
    """Đánh giá toàn bộ ảnh trong target_dir bằng Mask2Former Swin-S và lưu geneval_summary.csv"""
    if not target_dir or not os.path.exists(target_dir):
        return None

    geneval_csv_path = f"{target_dir}/geneval_summary.csv"
    if not FORCE_RESCORE and os.path.exists(geneval_csv_path):
        try:
            df_exist = pd.read_csv(geneval_csv_path)
            for _, r in df_exist.iterrows():
                if 'OVERALL' in str(r.iloc[0]).upper():
                    print(f"⚡ Tận dụng điểm GenEval đã chấm sẵn cho {exp_name}: {float(r.iloc[2]):.4f}")
                    return float(r.iloc[2])
        except Exception:
            pass

    # 1. Quét ảnh trong các thư mục prompt
    prompt_dir_map = {}
    for p_dir in glob.glob(f"{target_dir}/[0-9]*"):
        try:
            p_idx = int(os.path.basename(p_dir))
        except ValueError:
            continue
        imgs = sorted(glob.glob(f"{p_dir}/samples/*.png")) or [f for f in glob.glob(f"{p_dir}/*.png") if not f.endswith("grid.png")]
        if imgs:
            prompt_dir_map[p_idx] = imgs

    if not prompt_dir_map:
        print(f"⚠️ Không tìm thấy ảnh hợp lệ trong: {exp_name}")
        return None

    print(f"\n" + "="*75)
    print(f"🎯 Đang chấm GenEval (Mask2Former Swin-S): {exp_name}")
    print(f"   Số prompt có ảnh: {len(prompt_dir_map)}/553")
    print("="*75)

    task_results = {'single_object': [], 'two_object': [], 'counting': [], 'colors': [], 'position': [], 'color_attr': []}
    
    for p_idx in tqdm(sorted(prompt_dir_map.keys()), desc=f"Chấm điểm: {exp_name[:30]}..."):
        if p_idx >= len(prompts_meta):
            continue
        meta = prompts_meta[p_idx]
        tag = meta.get('tag', 'single_object')
        if tag not in task_results:
            continue

        imgs = prompt_dir_map[p_idx]
        prompt_scores = []
        for img_path in imgs:
            try:
                img = Image.open(img_path).convert('RGB')
                inputs = image_processor(images=img, return_tensors="pt").to(device)
                with torch.inference_mode():
                    outputs = detector(**inputs)

                # Instance Segmentation threshold = 0.5 chuẩn Mask2Former
                results = image_processor.post_process_instance_segmentation(
                    outputs, target_sizes=[img.size[::-1]], threshold=0.5
                )[0]
                segmentation = results["segmentation"].detach().cpu().numpy()
                segments_info = results["segments_info"]

                detected_objects = []
                for seg in segments_info:
                    c_name = id2label[seg["label_id"]].lower()
                    mask = (segmentation == seg["id"])
                    y_indices, x_indices = np.where(mask)
                    if len(x_indices) > 0 and len(y_indices) > 0:
                        x1, x2 = float(np.min(x_indices)), float(np.max(x_indices))
                        y1, y2 = float(np.min(y_indices)), float(np.max(y_indices))
                        box = [x1, y1, x2, y2]
                        center_x = (x1 + x2) / 2.0
                        center_y = (y1 + y2) / 2.0

                        if (x2 - x1 > 12 and y2 - y1 > 12):
                            crop = img.crop((max(0, x1), max(0, y1), min(img.width, x2), min(img.height, y2)))
                            pred_color = classify_crop_color(crop)
                        else:
                            pred_color = 'unknown'

                        detected_objects.append({
                            'class': c_name,
                            'box': box,
                            'center_x': center_x,
                            'center_y': center_y,
                            'color': pred_color
                        })

                # Đánh giá theo rule bài báo GenEval
                success = False
                includes = meta.get('include', [])
                if tag == 'single_object':
                    req_cls = includes[0]['class'].lower()
                    success = any(req_cls in obj['class'] or obj['class'] in req_cls for obj in detected_objects)
                elif tag == 'two_object':
                    req1, req2 = includes[0]['class'].lower(), includes[1]['class'].lower()
                    success = any(req1 in obj['class'] or obj['class'] in req1 for obj in detected_objects) and \
                               any(req2 in obj['class'] or obj['class'] in req2 for obj in detected_objects)
                elif tag == 'counting':
                    req_cls, target_count = includes[0]['class'].lower(), includes[0]['count']
                    found_count = sum(1 for obj in detected_objects if req_cls in obj['class'] or obj['class'] in req_cls)
                    success = (found_count == target_count)
                elif tag == 'colors':
                    req_cls, req_color = includes[0]['class'].lower(), includes[0]['color'].lower()
                    success = any((req_cls in obj['class'] or obj['class'] in req_cls) and (obj['color'] == req_color) for obj in detected_objects)
                elif tag == 'position':
                    req1, req2 = includes[0]['class'].lower(), includes[1]['class'].lower()
                    pos_type = includes[1].get('position', ['right of', 0])[0]
                    o1_list = [o for o in detected_objects if req1 in o['class'] or o['class'] in req1]
                    o2_list = [o for o in detected_objects if req2 in o['class'] or o['class'] in req2]
                    if o1_list and o2_list:
                        o1, o2 = o1_list[0], o2_list[0]
                        if 'right' in pos_type: success = (o2['center_x'] > o1['center_x'])
                        elif 'left' in pos_type: success = (o2['center_x'] < o1['center_x'])
                        elif 'above' in pos_type or 'top' in pos_type: success = (o2['center_y'] < o1['center_y'])
                        elif 'below' in pos_type or 'bottom' in pos_type: success = (o2['center_y'] > o1['center_y'])
                        else: success = True
                elif tag == 'color_attr':
                    success = all(any((inc['class'].lower() in obj['class'] or obj['class'] in inc['class'].lower()) and (obj['color'] == inc['color'].lower()) for obj in detected_objects) for inc in includes)

                prompt_scores.append(1.0 if success else 0.0)
            except Exception as e:
                pass

        if prompt_scores:
            task_results[tag].append(np.mean(prompt_scores))

    # Tổng hợp kết quả từng task
    summary_rows = []
    all_means = []
    for t_name, scores in task_results.items():
        mean_val = np.mean(scores) if scores else 0.0
        if scores: all_means.append(mean_val)
        summary_rows.append({'Nhiệm Vụ (Task)': t_name, 'Số Lượng Prompt': len(scores), 'Độ Chính Xác (Accuracy ↑)': f"{mean_val:.4f}"})

    overall_geneval = float(np.mean(all_means)) if all_means else 0.0
    summary_rows.append({'Nhiệm Vụ (Task)': '🔥 OVERALL GENEVAL BENCHMARK', 'Số Lượng Prompt': sum(len(s) for s in task_results.values()), 'Độ Chính Xác (Accuracy ↑)': f"{overall_geneval:.4f}"})
    
    df_geneval = pd.DataFrame(summary_rows)
    df_geneval.to_csv(geneval_csv_path, index=False)
    print(f"💾 Đã lưu bảng điểm GenEval chuẩn Mask2Former tại: {geneval_csv_path}")
    print(f"⭐ ĐIỂM OVERALL GENEVAL: {overall_geneval:.4f}")
    return overall_geneval

# Thực thi chấm điểm cho 6 thực nghiệm
for exp in matched_experiments:
    if exp["path"]:
        exp["geneval_score"] = evaluate_geneval_for_folder(exp["path"], exp["name"])
    else:
        exp["geneval_score"] = None

print("\n✅ ĐÃ HOÀN TẤT CHẤM ĐIỂM GENEVAL CHO TOÀN BỘ 6 THỰC NGHIỆM!")


## 6. Trích Xuất `final_metrics.json` & Xuất Bảng Tổng Hợp Đúng 6 Dòng

In [ ]:
# @title 📊 XUẤT BẢNG TỔNG HỢP KHOA HỌC CHUẨN ĐÚNG 6 DÒNG

def extract_metrics(folder_path):
    """Đọc các chỉ số ImageReward, CLIP, HPS v2.1 trực tiếp từ final_metrics.json"""
    if not folder_path or not os.path.exists(folder_path):
        return {"ir": "N/A", "clip": "N/A", "hps": "N/A", "count": 0}

    # 1. Đọc trực tiếp từ final_metrics.json (< 0.01s)
    fm_json = f"{folder_path}/final_metrics.json"
    if os.path.exists(fm_json):
        try:
            with open(fm_json, 'r') as f:
                d = json.load(f)
                ir_val = d.get('ImageReward', d.get('ir'))
                clip_val = d.get('CLIP', d.get('clip'))
                hps_val = d.get('HPS', d.get('hps'))
                count = d.get('total_prompts', len(glob.glob(f"{folder_path}/[0-9]*")))
                return {
                    "ir": f"{float(ir_val):.4f}" if ir_val is not None else "N/A",
                    "clip": f"{float(clip_val):.4f}" if clip_val is not None else "N/A",
                    "hps": f"{float(hps_val):.4f}" if hps_val is not None else "N/A",
                    "count": count
                }
        except Exception:
            pass

    # 2. Fallback quét results.json nếu final_metrics.json chưa được sinh
    r_files = glob.glob(f"{folder_path}/[0-9]*/results.json")
    if r_files:
        irs, clips, hpss = [], [], []
        for rf in r_files:
            try:
                with open(rf, 'r') as f:
                    d = json.load(f)
                if 'image_reward' in d: irs.append(d['image_reward'])
                if 'clip_score' in d: clips.append(d['clip_score'])
                if 'hps_score' in d: hpss.append(d['hps_score'])
            except Exception:
                pass
        return {
            "ir": f"{np.mean(irs):.4f}" if irs else "N/A",
            "clip": f"{np.mean(clips):.4f}" if clips else "N/A",
            "hps": f"{np.mean(hpss):.4f}" if hpss else "N/A",
            "count": len(r_files)
        }

    return {"ir": "N/A", "clip": "N/A", "hps": "N/A", "count": 0}

# Xây dựng đúng 6 dòng kết quả chuẩn
table_rows = []

for exp in matched_experiments:
    folder_path = exp["path"]
    metrics = extract_metrics(folder_path)
    
    # Lấy điểm GenEval chuẩn vừa chấm
    ge_val = exp.get("geneval_score")
    if ge_val is None and folder_path:
        # Thử đọc từ geneval_summary.csv nếu có
        ge_csv = f"{folder_path}/geneval_summary.csv"
        if os.path.exists(ge_csv):
            try:
                df_ge = pd.read_csv(ge_csv)
                for _, r in df_ge.iterrows():
                    if 'OVERALL' in str(r.iloc[0]).upper():
                        ge_val = float(r.iloc[2])
                        break
            except Exception:
                pass
                
    ge_str = f"{ge_val:.4f}" if ge_val is not None else "N/A"
    method_name = f"🔥 {exp['method']}" if "RS-LiDAR" in exp["method"] else exp["method"]
    
    table_rows.append({
        "Setting (Cấu Hình)": exp["setting"],
        "Phương Pháp": method_name,
        "ImageReward ↑": metrics["ir"],
        "CLIP-Score ↑": metrics["clip"],
        "HPS v2.1 ↑": metrics["hps"],
        "GenEval ↑": ge_str,
        "Số Prompts": f"{metrics['count']}/553",
        "Thư Mục Dữ Liệu": exp["name"]
    })

final_6rows_df = pd.DataFrame(table_rows)

print("\n" + "="*110)
print("📊 BẢNG KẾT QUẢ KHOA HỌC CHUẨN 6 DÒNG (3 SETTINGS x 2 METHODS)")
print("   Mô hình đánh giá GenEval: Mask2Former Swin-S COCO (Threshold 0.5) + CLIP ViT-B/32")
print("="*110)
display(final_6rows_df)

# Tính toán mức tăng trưởng Delta (Δ) của RS-LiDAR so với Vanilla LiDAR trên từng Setting
print("\n" + "="*110)
print("📈 PHÂN TÍCH MỨC TĂNG TRƯỞNG (Δ GAIN) CỦA RS-LiDAR SO VỚI VANILLA LIDAR:")
print("="*110)

for s_idx in [0, 2, 4]:
    row_lidar = table_rows[s_idx]
    row_rslidar = table_rows[s_idx + 1]
    setting_name = row_lidar["Setting (Cấu Hình)"]
    
    try:
        delta_ir = float(row_rslidar["ImageReward ↑"]) - float(row_lidar["ImageReward ↑"])
        pct_ir = (delta_ir / abs(float(row_lidar["ImageReward ↑"]))) * 100
        delta_clip = float(row_rslidar["CLIP-Score ↑"]) - float(row_lidar["CLIP-Score ↑"])
        delta_hps = float(row_rslidar["HPS v2.1 ↑"]) - float(row_lidar["HPS v2.1 ↑"])
        delta_ge = float(row_rslidar["GenEval ↑"]) - float(row_lidar["GenEval ↑"])
        
        print(f"🔹 [{setting_name}]:")
        print(f"   • ImageReward: {delta_ir:+.4f} ({pct_ir:+.2f}%)")
        print(f"   • GenEval:     {delta_ge:+.4f}")
        print(f"   • CLIP-Score:  {delta_clip:+.4f}")
        print(f"   • HPS v2.1:    {delta_hps:+.4f}")
    except Exception as e:
        print(f"🔹 [{setting_name}]: Chưa đủ dữ liệu số để tính delta ({e})")
    print("-" * 70)

# Lưu ra Google Drive
master_csv_path = f"{TARGET_BASE}/table2_master_6rows_summary.csv"
final_6rows_df.to_csv(master_csv_path, index=False)
print(f"\n💾 Đã lưu bảng tổng hợp 6 dòng ra Google Drive tại:\n   {master_csv_path}")
print("="*110)
